In [1]:
import cv2
import os
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from collections import Counter
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Absolute directory paths (using raw strings r"...")
train_dir = r'C:\Users\sagal\Desktop\Let us build\RAF-DB\DATASET\train'
test_dir  = r'C:\Users\sagal\Desktop\Let us build\RAF-DB\DATASET\test'

# Reproducibility seed
random.seed(42)
torch.manual_seed(42)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=train_dir, transform=val_transform)

print("Classes:", train_dataset.classes)
print("Total train images:", len(train_dataset))

Classes: ['1', '2', '3', '4', '5', '6', '7']
Total train images: 12271


In [6]:
indices = list(range(len(train_dataset)))
random.seed(42)
random.shuffle(indices)

split_size    = int(0.85 * len(indices))
train_indices = indices[:split_size]
val_indices   = indices[split_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset   = Subset(val_dataset,   val_indices)

# Weighted Sampler logic
train_labels = [train_dataset.targets[i] for i in train_indices]
class_counts = Counter(train_labels)

total = len(train_labels)
class_weights = {cls: total / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_subset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_subset,   batch_size=32, shuffle=False)

print(f"Train size: {len(train_subset)} | Val size: {len(val_subset)}")

Train size: 10430 | Val size: 1841


In [7]:
resnet = models.resnet18(weights="IMAGENET1K_V1")

# Unfreeze layer3, layer4, and fc
for name, param in resnet.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

resnet.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(resnet.fc.in_features, 7)
)

model = resnet.to(device)

# Quick sanity check
dummy = torch.randn(32, 3, 224, 224).to(device)
print("Output shape:", model(dummy).shape)

Output shape: torch.Size([32, 7])


In [8]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = Adam([
    {"params": resnet.layer3.parameters(), "lr": 1e-6, "weight_decay": 1e-4},
    {"params": resnet.layer4.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    {"params": resnet.fc.parameters(),     "lr": 1e-4, "weight_decay": 1e-3}
])

num_epochs       = 40
patience         = 10
best_val_loss    = float("inf")
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc  = train_correct / len(train_subset) * 100
    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_acc  = val_correct / len(val_subset) * 100
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_rafdb_resnet18_regularized.pth")
        print(f"  ✓ Best model saved (val loss: {val_loss:.4f}, val acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_rafdb_resnet18_regularized.pth"))
print("Best regularized model loaded!")

Epoch [1/40]  Train Loss: 1.9579  Train Acc: 23.84%  |  Val Loss: 1.7519  Val Acc: 33.62%
  ✓ Best model saved (val loss: 1.7519, val acc: 33.62%)
Epoch [2/40]  Train Loss: 1.7525  Train Acc: 34.75%  |  Val Loss: 1.6240  Val Acc: 41.61%
  ✓ Best model saved (val loss: 1.6240, val acc: 41.61%)
Epoch [3/40]  Train Loss: 1.6365  Train Acc: 40.20%  |  Val Loss: 1.5361  Val Acc: 46.88%
  ✓ Best model saved (val loss: 1.5361, val acc: 46.88%)
Epoch [4/40]  Train Loss: 1.5365  Train Acc: 46.50%  |  Val Loss: 1.4600  Val Acc: 51.49%
  ✓ Best model saved (val loss: 1.4600, val acc: 51.49%)
Epoch [5/40]  Train Loss: 1.4732  Train Acc: 49.46%  |  Val Loss: 1.4184  Val Acc: 53.61%
  ✓ Best model saved (val loss: 1.4184, val acc: 53.61%)
Epoch [6/40]  Train Loss: 1.4242  Train Acc: 52.57%  |  Val Loss: 1.3789  Val Acc: 55.35%
  ✓ Best model saved (val loss: 1.3789, val acc: 55.35%)
Epoch [7/40]  Train Loss: 1.3782  Train Acc: 54.77%  |  Val Loss: 1.3521  Val Acc: 57.90%
  ✓ Best model saved (val lo

In this experiment, we addressed the severe overfitting in our facial emotion classifier by implementing a fully regularized PyTorch pipeline. We introduced data augmentations—including random affine transformations, color jittering, and random erasing—to prevent the model from memorizing individual image pixels. We also increased the classification head's dropout to 0.5, added weight decay to the Adam optimizer, and applied label smoothing inside the loss function while tracking lowest validation loss. As a result, we successfully closed the overfitting gap from a 19% split down to an 8% split—maintaining our ~74% validation accuracy while ensuring the network is now learning true, generalizable facial representations rather than dataset-specific noise.